# Transformer Linear Layer Performance Predictor

## Run a benchmark session

This notebook embeds release **1.0.0**, including the fixed experiment plans.
Select **Runtime > Change runtime type > GPU** before a measured run. The recorded
hardware identity, rather than the requested allocation, determines the device used.

The workload is preallocated `Y = X @ W.T`. The primary label is a warm-reuse
CUDA Graph replay interval divided by its GEMM count. GPU reads and writes are
inside the interval. Host transfers, graph capture, bias, activation, and serving
queues are outside it. This is not an end-to-end language-model benchmark.

Keep every result archive, including failed or qualified runs. The launcher makes
one attempt. The campaign has a 30-minute budget and the worker a 32-minute timeout;
dependency setup and export are additional. It retains the runtime's PyTorch/CUDA.


In [ ]:
# Choose a new session ID for each attempt; existing output is never overwritten.
ACTION = "run"               # "run" or "diagnose"; diagnose does not open held-out data.
DTYPE = "float16"            # "float16" or "bfloat16"; fit each dtype separately.
SESSION_ID = "fp16_s01"
SIMULATE = False             # True exercises software on CPU; it is not a GPU measurement.
PIN_ANALYSIS_DEPS = True     # Pin the recorded NumPy/SciPy/Matplotlib analysis stack.


## Load the release
This verifies the embedded source and Python syntax.

In [ ]:
# Verify and load the embedded release; this cell performs no GPU measurement.
import base64, hashlib, types
_LAUNCHER_B64 = '...'
_launcher_source = base64.b64decode(_LAUNCHER_B64).decode('utf-8')
assert hashlib.sha256(_launcher_source.encode()).hexdigest() == '0a18b1eaa07dceecfb2fd29bb141b2fa686d8fb9abef5d23e02dd2162e66094e', 'Launcher integrity mismatch'
launcher = types.ModuleType('linear_perf_colab')
exec(compile(_launcher_source, 'linear_perf_colab.py', 'exec'), launcher.__dict__)
print(launcher.self_check())


## Execute one session

Progress and quality flags are retained. Download the archive as soon as the run
finishes: a complete Colab runtime loss can prevent any program from exporting.


In [ ]:
result = launcher.run(session_id=SESSION_ID, dtype=DTYPE, action=ACTION,
                      simulate=SIMULATE, pin_analysis_deps=PIN_ANALYSIS_DEPS)
print("Result archive:", result["archive"])


## Download the complete result

For a completed run, open `run/report.html` after extraction. For diagnostics or
failures, read `launcher_status.json`, `worker.log`, and `run/status.json`.


In [ ]:
from google.colab import files
files.download(result["archive"])
files.download(result["archive"] + ".sha256")


## Plan repeated sessions before looking at test scores

- A paired dtype comparison requires the same recorded physical GPU UUID. Run
  `bfloat16` with `bf16_s01` in the same runtime, then download both archives.
- Repeat the registered procedure with session IDs ending in `s02` and `s03`.
  Separate processes do not establish separate physical devices.
- Use `Review_Results.ipynb` for CPU-only auditing and comparisons of archives
  produced by this exact embedded release. Fix the transfer source in advance.
- Inspect the measurement kind, formal selected model, test and challenge errors,
  tail errors, controls, and quality flags. Synthetic data are software checks only.

The repository's published A100 snapshot has its own offline verification command:
`python scripts/verify_published_results.py`. It is not an input archive for this launcher.
